In [1]:
import importlib
import json, rich
import spacy
import models.labelstudio_models
importlib.reload(models.labelstudio_models)

<module 'models.labelstudio_models' from '/home/ronji/repos/CarP/ner-spacy-v2/models/labelstudio_models.py'>

In [ ]:
from models.labelstudio_models import Data,Value,Result,Prediction,Document
from pydantic import BaseModel

### reformat into labelstudio import file for revision

In [ ]:
with open('cleaned_resumes.json','r') as j:
    labeled_resumes = json.loads(j.read())

In [ ]:
documents = []

for r in labeled_resumes:
    name, text, labels = r['id'], r['content'], r['labels']
    
    results = []
    for l in labels:
        start, end, label, value = l["start"], l["end"], l["label"], l["value"]
        results.append(Result.create(start, end, label, value))
    
    documents.append(Document.create(text, results).model_dump())
    
with open('export_labelstudio.json', 'w') as f:
 
    f.write(json.dumps(documents))

In [ ]:
### formatting of spacy entity patterns file for skillNer

In [15]:
with open('skill_db_relax_20.json', 'r') as f:
    skill_db = json.loads(f.read())

In [ ]:
skills = list(skill_db.values())
rich.print(skills[0])
print(len(skills))

In [ ]:
nlp.disable_pipe('entity_ruler')

In [17]:
nlp = spacy.load('spacy_outputs/model-last')
ruler = nlp.add_pipe('entity_ruler')

In [18]:
patterns = []
for s in skills:
    label = s['skill_type']
    if label == "Certification": 
        continue
    pattern = str(s['skill_name']).lower()
    p_list = pattern.split(' ')
    p_dict = []
    for p in p_list:
        p_dict.append({"LOWER":p})
    patterns.append({
        'label':label,
        'pattern':p_dict
        })

In [19]:
print(len(patterns))

29392


In [20]:
ruler.add_patterns(patterns)

In [21]:
ruler.to_disk("patterns.jsonl")